# Lesson 4.4: Preparing the Collaborative Review Sheet

> ⚠️ **This is an instructor notebook.** Run it once to generate the Google Sheet CSV. Students do not need to open or run this file.

---

The geoparser produced 1,784 rows of resolved locations across 522 unique place names. Before we can map anything meaningfully, a human review is needed to catch the six failure types from Lesson 4.3:

| Failure | Example |
|---|---|
| False positive | `Virginia` tagged as separate destination |
| Imprecise resolution | `Shenandoah Valley` pinned to a specific building |
| False negative | `Hburg` missed entirely |
| Ambiguous abbreviation | `Lex` not recognized |
| Hyperlocal | `his house` unresolvable |
| Wrong disambiguation | `JMU` → Jiamusi, China |

This notebook:
1. Deduplicates 1,784 rows to unique places
2. Enriches each place with GeoNames metadata (type, state, country, population)
3. Assigns each place to a student group (balanced by mention count, not just letter count)
4. Exports a review-ready CSV for Google Sheets

## Step 1: Setup

In [1]:
import pandas as pd
import sqlite3
import pathlib
import platform

# ── Locate GeoNames SQLite database ──────────────────────────────────────────
if platform.system() == 'Windows':
    GEONAMES_DB = pathlib.Path.home() / 'AppData' / 'Local' / 'geoparser' / 'geonames' / 'geonames.db'
else:
    GEONAMES_DB = pathlib.Path.home() / '.local' / 'share' / 'geoparser' / 'geonames' / 'geonames.db'

if not GEONAMES_DB.exists():
    raise FileNotFoundError(
        f"GeoNames database not found at {GEONAMES_DB}.\n"
        "Run: python -m geoparser download geonames"
    )

print(f"✅ GeoNames DB found: {GEONAMES_DB}")

✅ GeoNames DB found: C:\Users\joost\AppData\Local\geoparser\geonames\geonames.db


## Step 2: Load and Deduplicate

In [2]:
df_raw = pd.read_csv('data/jmu_reddit_geoparsed_long.csv')
print(f"Raw rows: {len(df_raw):,}  |  Unique places: {df_raw['place'].nunique():,}")

# Collect up to 3 unique sample sentences per place
def collect_samples(sentences, n=3, max_len=160):
    unique = list(dict.fromkeys(str(s).strip() for s in sentences if pd.notna(s)))
    return ' | '.join(s[:max_len] + ('…' if len(s) > max_len else '') for s in unique[:n])

df_unique = (
    df_raw.groupby('place', sort=True)
    .agg(
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
        count=('place', 'size'),
        sample_posts=('sentences', collect_samples),
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f"\nDeduped to {len(df_unique):,} unique places")
print(f"  Mentioned 10+ times:  {(df_unique['count'] >= 10).sum():>4}  ← high priority")
print(f"  Mentioned  5–9 times: {((df_unique['count'] >= 5) & (df_unique['count'] < 10)).sum():>4}")
print(f"  Mentioned  2–4 times: {((df_unique['count'] >= 2) & (df_unique['count'] < 5)).sum():>4}")
print(f"  Mentioned  1 time:    {(df_unique['count'] == 1).sum():>4}  ← low priority")

Raw rows: 1,784  |  Unique places: 522

Deduped to 522 unique places
  Mentioned 10+ times:    25  ← high priority
  Mentioned  5–9 times:   37
  Mentioned  2–4 times:  146
  Mentioned  1 time:     314  ← low priority


## Step 3: Enrich with GeoNames Metadata

For each resolved place, we look up its `feature_type` (e.g. "populated place", "state", "country", "school"), the US state it's in, its country, and its population. This extra context helps students make faster decisions in the review.

In [ ]:
def lookup_geonames(lat, lon, con, tol=0.005):
    """Look up GeoNames metadata by coordinates."""
    cur = con.cursor()
    cur.execute(
        '''SELECT feature_type, admin1_name, country_name, population
           FROM locations
           WHERE latitude BETWEEN ? AND ?
             AND longitude BETWEEN ? AND ?
           ORDER BY population DESC
           LIMIT 1''',
        (lat - tol, lat + tol, lon - tol, lon + tol)
    )
    row = cur.fetchone()
    return row if row else (None, None, None, None)

print("Looking up GeoNames metadata for each unique place...")
con = sqlite3.connect(GEONAMES_DB)

results = [
    lookup_geonames(row.latitude, row.longitude, con)
    for row in df_unique.itertuples()
]

con.close()

df_unique[['feature_type', 'state', 'country', 'population']] = pd.DataFrame(
    results, columns=['feature_type', 'state', 'country', 'population']
)

print(f"✅ Enrichment complete")
print(f"\nTop feature types:")
print(df_unique['feature_type'].value_counts().head(15).to_string())

Looking up GeoNames metadata for each unique place...


KeyboardInterrupt: 

: 

## Step 4: Assign Student Groups

We need 5 groups with **roughly equal review workload** (weighted by mention count, not just number of places). A greedy algorithm sorts places alphabetically then assigns them to whichever group currently has the lowest total mention count.

In [ ]:
N_GROUPS = 5

# Sort alphabetically for assignment
df_alpha = df_unique.sort_values('place').reset_index(drop=True)

# Greedy bin-packing: assign each place to the group with the current lowest total
group_totals = [0] * N_GROUPS
group_labels = []

for count in df_alpha['count']:
    g = group_totals.index(min(group_totals))
    group_labels.append(g + 1)  # 1-indexed
    group_totals[g] += count

df_alpha['assigned_group'] = group_labels

# Build human-readable letter ranges for each group
group_info = []
for g in range(1, N_GROUPS + 1):
    subset = df_alpha[df_alpha['assigned_group'] == g]['place']
    first_letters = sorted(set(str(p)[0].upper() for p in subset))
    letter_range = f"{first_letters[0]}–{first_letters[-1]}"
    group_info.append({
        'Group': g,
        'Letter range': letter_range,
        'Places': len(subset),
        'Total mentions': group_totals[g - 1],
    })

summary = pd.DataFrame(group_info)
print("Group assignment summary:")
print(summary.to_string(index=False))

## Step 5: Export Review Sheet

In [ ]:
# Build the final review sheet with reference columns + empty review columns
df_review = df_alpha[[
    'place', 'feature_type', 'state', 'country', 'population',
    'count', 'latitude', 'longitude', 'sample_posts', 'assigned_group'
]].copy()

# Add empty columns for students to fill in
df_review['action']         = ''   # KEEP / CORRECT / REMOVE
df_review['corrected_name'] = ''   # Only if action = CORRECT
df_review['corrected_lat']  = ''   # Only for campus/hyperlocal places
df_review['corrected_lon']  = ''   # Only for campus/hyperlocal places
df_review['reviewer']       = ''   # Student name

# Sort: within each group, high-count places first
df_review = df_review.sort_values(
    ['assigned_group', 'count'], ascending=[True, False]
).reset_index(drop=True)

out_path = 'data/jmu_reddit_geoparsed_review.csv'
df_review.to_csv(out_path, index=False)

print(f"✅ Saved {out_path}")
print(f"   {len(df_review):,} rows  |  {len(df_review.columns)} columns")
print(f"\nColumn order:")
for i, col in enumerate(df_review.columns, 1):
    filled = '← students fill this' if col in ('action','corrected_name','corrected_lat','corrected_lon','reviewer') else ''
    print(f"  {i:>2}. {col:<20} {filled}")

## Step 6: Preview

In [ ]:
# Show a representative sample — one high-priority and one problem row per group
pd.set_option('display.max_colwidth', 80)
display(
    df_review[['place','feature_type','state','country','count','assigned_group']]
    .head(30)
)

---

## Google Sheets Setup

### 1 — Upload
1. Go to [sheets.google.com](https://sheets.google.com) → **New spreadsheet**
2. **File → Import → Upload** → select `data/jmu_reddit_geoparsed_review.csv`
3. Rename the sheet: *JMU Reddit Location Review*
4. Share with the class ("Anyone with the link can edit")

### 2 — Freeze and protect reference columns
1. **View → Freeze → 1 row** (locks the header)
2. Select columns A–J (the reference data) → right-click → **Protect range** → set to "Show a warning when editing"

### 3 — Data validation on `action` (column K)
1. Select the entire column K (below the header)
2. **Data → Data validation → Add rule**
3. Criteria: **Dropdown** → enter: `KEEP`, `CORRECT`, `REMOVE`
4. Under "If the data is invalid": **Reject the input**
5. Add a note in cell K1: *Required. KEEP = correct as-is. CORRECT = right place, wrong name/coords. REMOVE = not a real place.*

### 4 — Conditional formatting
Apply these rules to columns A–O:

| Rule | Format |
|---|---|
| Column K = `REMOVE` | Red background |
| Column K = `CORRECT` | Yellow background |
| Column K = `KEEP` | Green background |
| Column J = `1` | Light blue row tint |
| Column J = `2` | Light green row tint |
| Column J = `3` | Light yellow row tint |
| Column J = `4` | Light orange row tint |
| Column J = `5` | Light purple row tint |

### 5 — Create a filter view per group
1. **Data → Create a filter view** → name it *Group 1*
2. Click the filter icon on column J (`assigned_group`) → filter to `1`
3. Repeat for groups 2–5
4. Each student team uses their group's filter view — they only see their rows

### 6 — Student instructions (paste into a second sheet tab)

```
HOW TO REVIEW YOUR ROWS
========================
1. Switch to your group's filter view (Data → Filter views → Group N)
2. For each row, read the 'sample_posts' column and decide:
   - KEEP   → the geoparser got it right
   - CORRECT → it found a real place but got the name or coordinates wrong
              (fill in corrected_name; add corrected_lat/lon for campus buildings)
   - REMOVE  → it's not actually a place (FCS, Breeze, his house, etc.)
3. Enter your name in the 'reviewer' column for every row you touch
4. If you're unsure, search the place in Google Maps in the context of JMU
5. For campus buildings, search "[name] JMU" in Google Maps for exact coordinates

TIPS
====
- 'count' = how often it appears in the data. Higher count = more impact on the map
- 'feature_type' = what GeoNames thinks the place is. 'populated place' is usually fine;
  'school' or 'building' may be a campus location needing corrected coordinates
- Single-mention places (count = 1) are lower priority — do these last
```

### 7 — Downloading the finished sheet
When all groups are done:
**File → Download → Comma-separated values (.csv)**

Save as `data/jmu_reddit_geoparsed_reviewed.csv` and run `lesson_4_5_applying_corrections.ipynb`.